# Ensemble-Only: Berhat'ın 84.54 modeli üstünde WBS + TTA + Multi-model

**Bu notebook eğitim YAPMAZ** — sadece mevcut V3-augmented model üstünde ensemble çalıştırır (~20 dk).

## Gerekli 2 Input (Add Data):

1. **IAM word dataset** — `words.txt` + `words/` içeren herhangi bir IAM word dataset (örn: `iam-handwriting-word-database`)
2. **Model dosyası** — 2 yoldan biri:
   - **Yol A (Kaggle Dataset):** `best_model_wa.pth` dosyasını Kaggle Dataset olarak yükle (private OK) → Add Input
   - **Yol B (Notebook Output):** Add Data → Notebook Output → 84.54 çıkardığın önceki notebook'unu seç

Notebook otomatik `/kaggle/input/` altında `best_model_wa.pth` arar.

## Settings:
- Accelerator: **GPU T4 x1** (aynı precision reprodüksiyonu için)
- Internet: **ON** (git clone + WBS pip source install)

## Çıktı:
- `results/ensemble_results.json` — 12 strategy × WA + Wilson CI + McNemar
- Beklenen best: **~%86-87** (V3 + WBS veya V2+V3+WBS)

In [ ]:
# Hücre 1: GPU + donanım
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.0f} GB")
    print(f"PyTorch: {torch.__version__}")
!python --version

In [ ]:
# Hücre 2: GitHub'dan repo clone + WBS install + verify
import sys, os, subprocess, shutil

REPO_URL = "https://github.com/Ridvan013/CRNN-Handwriting-Recognition.git"
BRANCH   = "feature/aachen-v3-extended-trigram"
REPO_DIR = "/kaggle/working/repo"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
    check=True,
)
print(f"✓ Repo klonlandı")

# Kod + V1/V2 modelleri + ensemble script'i /kaggle/working'e taşı
for name in ["cloud", "aachen_splits", "trigram_lm.py",
             "Model_aachen", "Model_aachen_v2",
             "ensemble_v2v3_and_tta.py", "ensemble_inference.py"]:
    src = os.path.join(REPO_DIR, name)
    dst = os.path.join("/kaggle/working", name)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif os.path.isfile(src):
        shutil.copy(src, dst)
    print(f"  {'✓' if os.path.exists(dst) else '⚠'} {name}")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")

# WBS install (source-build, Py 3.12 uyumlu)
!pip install --quiet git+https://github.com/githubharald/CTCWordBeamSearch.git

# NLTK words
import nltk
try:
    nltk.data.find("corpora/words")
except LookupError:
    nltk.download("words", quiet=True)

# WBS verify — sessiz fail yok
try:
    import word_beam_search
    from word_beam_search import WordBeamSearch
    print(f"✓ WBS kuruldu: {word_beam_search.__file__}")
except Exception as e:
    raise RuntimeError(f"WBS kurulumu BAŞARISIZ: {e}\nSettings → Internet: ON olduğundan emin ol.")

In [ ]:
# Hücre 3: Model dosyasını /kaggle/input/ altında ara
import os, subprocess

print("=== /kaggle/input altındaki datasetler ===")
!ls /kaggle/input/
print()

print("=== best_model_wa.pth aranıyor... ===")
result = subprocess.run(
    ["find", "/kaggle/input", "-name", "best_model_wa.pth", "-type", "f"],
    capture_output=True, text=True
)
found = [p.strip() for p in result.stdout.strip().splitlines() if p.strip()]
for p in found:
    size_mb = os.path.getsize(p) / 1024**2
    print(f"  {p}  ({size_mb:.1f} MB)")

# En büyük dosyayı V3-augmented modeli olarak seç (110 MB civarı)
if not found:
    raise RuntimeError(
        "best_model_wa.pth bulunamadı!\n"
        "Sağ panel → Add Input → önceki notebook output'unu ekle veya .pth'i dataset olarak yükle"
    )

# V3 augmented (~110 MB) — en büyük
V3_MODEL_PATH = max(found, key=os.path.getsize)
print(f"\n✓ V3-augmented model seçildi: {V3_MODEL_PATH}")
print(f"  ({os.path.getsize(V3_MODEL_PATH)/1024**2:.1f} MB)")

# Sanity check: model gerçekten V3 mimari mi?
sd = torch.load(V3_MODEL_PATH, map_location="cpu", weights_only=True)
n_params = sum(v.numel() for v in sd.values() if hasattr(v, "numel"))
print(f"  Params: {n_params:,} (beklenen ~28.7M)")
assert abs(n_params - 28_735_313) < 1000, f"Beklenmeyen model boyutu: {n_params}"

In [ ]:
# Hücre 4: IAM words.txt + words/ dizinini bul
import subprocess, os

result = subprocess.run(
    ["find", "/kaggle/input", "-name", "words.txt", "-maxdepth", "6"],
    capture_output=True, text=True
)
found_words = [p.strip() for p in result.stdout.strip().splitlines() if p.strip()]

IAM_WORDS_TXT = None
IAM_WORDS_DIR = None
for wt in found_words:
    candidate_dir = os.path.join(os.path.dirname(wt), "words")
    if os.path.isdir(candidate_dir):
        IAM_WORDS_TXT = wt
        IAM_WORDS_DIR = candidate_dir
        break

if not IAM_WORDS_TXT or not IAM_WORDS_DIR:
    raise RuntimeError("IAM words.txt veya words/ dizini bulunamadı — IAM word dataset ekle")
print(f"✓ IAM words.txt: {IAM_WORDS_TXT}")
print(f"✓ IAM words/  : {IAM_WORDS_DIR}")

In [ ]:
# Hücre 5: Ensemble script'i çalıştır (V1 + V2 + Berhat V3 + WBS + TTA)
# Kaggle T4 üstünde AMP autocast precision uyumlu → gerçek WBS+ensemble sonucu

!python ensemble_v2v3_and_tta.py \
    --v1 Model_aachen/best_model_wa.pth \
    --v2 Model_aachen_v2/best_model_wa.pth \
    --v3 {V3_MODEL_PATH} \
    --iam-words {IAM_WORDS_TXT} \
    --iam-root {IAM_WORDS_DIR} \
    --output-json /kaggle/working/results/ensemble_results.json \
    --baseline-strategy "V3-aug + Trigram"

In [ ]:
# Hücre 6: Sonuçları oku ve özetle
import json

p = "/kaggle/working/results/ensemble_results.json"
if os.path.exists(p):
    r = json.load(open(p))
    print("=" * 78)
    print(f"  ENSEMBLE STRATEGIES ON AACHEN TEST (N={r['n_samples']:,})")
    print("=" * 78)
    print(f"  {'Strategy':<38s} {'WA':>8s} {'Wilson 95% CI':>22s}")
    print(f"  {'-'*38} {'-'*8} {'-'*22}")
    for s in r["strategies"]:
        n = s["name"]; wa = s["wa_pct"]
        ci = s["wilson_95ci_pct"]
        print(f"  {n:<38s} {wa:7.2f}%  [{ci[0]:5.2f}%, {ci[1]:5.2f}%]")

    best = r["best_strategy"]
    print("=" * 78)
    print(f"  BEST: {best['name']}  →  {best['wa_pct']:.2f}%  "
          f"CI[{best['wilson_95ci_pct'][0]:.2f}, {best['wilson_95ci_pct'][1]:.2f}]")
    print("=" * 78)

    print("\n--- Berhat baseline karşılaştırma ---")
    print(f"V3-aug + Trigram (baseline): 84.54% [83.55%, 85.49%]")
    print(f"Ensemble best:               {best['wa_pct']:.2f}% "
          f"[{best['wilson_95ci_pct'][0]:.2f}%, {best['wilson_95ci_pct'][1]:.2f}%]")
    delta = best['wa_pct'] - 84.54
    print(f"Delta:                       {delta:+.2f}pp")

    print(f"\n--- McNemar tests (anlamlı iyileşmeler) ---")
    for row in r["mcnemar_vs_baseline"]:
        if row["significant_p01"] and row["delta_pp"] > 0:
            print(f"  {row['strategy']:<38s} Δ={row['delta_pp']:+.2f}pp  p={row['p_exact']:.2e}  ✓")
else:
    print(f"⚠️ {p} yok — ensemble çalışmadı")